# Lab 2: Vector Search Foundations

## with Oracle AI Database 26ai and LangChain OracleVS

--------

## Step 1: Connect to Oracle AI Database

Your environment has been pre-configured with Oracle AI Database 26ai running locally. The `VECTOR` user and connection details are ready to use.

In [ ]:
import oracledb
import time
import logging

%store -r adb_dsn vector_user vector_password

def connect_to_oracle(
    max_retries=3,
    retry_delay=5,
    user=vector_user,
    password=vector_password,
    dsn=adb_dsn,
    program="seergroup.proteus.workshop",
):
    """
    Connect to Oracle database with retry logic and helpful error messages.
    """
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user, password=password, dsn=dsn, program=program
            )
            print("✓ Connected successfully!")

            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%'")
                banner = cur.fetchone()[0]
                print(f"\n{banner}")

            return conn

        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")

            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → Database may still be starting. Retrying...")
                if attempt < max_retries:
                    time.sleep(retry_delay)
                else:
                    raise
            else:
                raise

    raise ConnectionError("Failed to connect after all retries")

vector_conn = connect_to_oracle()
print("Using user:", vector_conn.username)



--------

## Step 2: Initialize Embeddings and Create a Vector-Enabled Table

We'll use the `sentence-transformers/paraphrase-mpnet-base-v2` model to convert text into 768-dimensional vectors. OracleVS handles the table creation, embedding storage, and similarity search under the hood.

In [ ]:
from langchain_oracledb.vectorstores import OracleVS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores.oraclevs import create_index
from langchain_community.vectorstores.utils import DistanceStrategy

# Suppress verbose logging from langchain_oracledb
logging.getLogger("langchain_oracledb").setLevel(logging.CRITICAL)

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

# Create the vector-enabled SQL table via OracleVS
vector_store = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name="VECTOR_SEARCH_DEMO",
    distance_strategy=DistanceStrategy.COSINE,
)


### Create an HNSW Index

HNSW (Hierarchical Navigable Small World) is a graph-based approximate nearest-neighbor index. It provides fast, accurate similarity search at scale — essential when your knowledge base grows to thousands or millions of documents.

In [ ]:
def safe_create_index(conn, vs, idx_name):
    """Create index, skipping if it already exists."""
    try:
        create_index(
            client=conn,
            vector_store=vs,
            params={"idx_name": idx_name, "idx_type": "HNSW"},
        )
        print(f"  ✅ Created index: {idx_name}")
    except Exception as e:
        if "ORA-00955" in str(e):
            print(f"  ⏭️ Index already exists: {idx_name} (skipped)")
        else:
            raise


safe_create_index(vector_conn, vector_store, "oravs_hnsw")


--------

## Step 3: Ingest Research Papers

In a real deployment, this data would come from institutional repositories, journal APIs, or preprint servers. Here we'll load papers from the `nick007x/arxiv-papers` dataset on HuggingFace — a collection of arXiv papers spanning multiple disciplines that Proteus will use to answer research queries.

In [ ]:
from datasets import load_dataset

MAX_PAPERS = 300
ds_stream = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

sampled_papers = []
texts = []
metadata_list = []

for i, item in enumerate(ds_stream):
    if i >= MAX_PAPERS:
        break

    arxiv_id = item.get("arxiv_id", f"unknown_{i}")
    title = (item.get("title") or "").strip()
    abstract = (item.get("abstract") or "").strip()
    primary_subject = (item.get("primary_subject") or "").strip()
    authors = item.get("authors") or []

    if isinstance(authors, str):
        authors_text = authors
    elif isinstance(authors, list):
        authors_text = ", ".join(str(a).strip() for a in authors if str(a).strip())
    else:
        authors_text = ""

    text = f"Title: {title}\nAbstract: {abstract}"

    sampled_papers.append({
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "primary_subject": primary_subject,
        "authors": authors_text,
    })
    texts.append(text)
    metadata_list.append({
        "id": arxiv_id,
        "arxiv_id": arxiv_id,
        "title": title,
        "primary_subject": primary_subject,
        "authors": authors_text,
    })

vector_store.add_texts(texts=texts, metadatas=metadata_list)
print(f"✅ Ingested {len(texts)} research papers into VECTOR_SEARCH_DEMO")

In [ ]:
# Inspect one sample to see the metadata fields we can filter on
sample_primary_subject = sampled_papers[0]["primary_subject"] if sampled_papers else ""
sample_arxiv_id = sampled_papers[0]["arxiv_id"] if sampled_papers else ""
print("Sample primary subject:", sample_primary_subject)
print("Sample arxiv_id:", sample_arxiv_id)



--------

## Step 4: Querying with Natural Language

Now let's see the power of semantic search. Unlike keyword search, vector similarity finds documents based on *meaning*. A query about "how neural networks learn representations" will match papers about deep learning, feature extraction, and representation learning — even if those exact words don't appear in the query.

### Basic Similarity Search

In [ ]:
query = "Find research papers about planetary exploration mission planning"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print("Title:", doc.metadata.get("title", "N/A"))
    print("Subject:", doc.metadata.get("primary_subject", "N/A"))
    print("Text:", doc.page_content[:200], "...")
    print()


> **🔍 Notice** how the search returns relevant papers even when the query uses different terminology than the paper titles. That's semantic similarity at work — the embedding model understands the *meaning* behind the words.

### Search with Relevance Scores

Scores let Proteus gauge confidence. A score close to 0 means high similarity (cosine distance); a score near 1 means low relevance.

In [ ]:
query = "methods for detecting gravitational waves"

results = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results:
    print(f"Score: {score:.4f}")
    print(f"Title: {doc.metadata.get('title', 'N/A')}")
    print(f"Subject: {doc.metadata.get('primary_subject', 'N/A')}")
    print("------")


--------

## Step 5: Filtered Search with Metadata

In a real research workflow, Proteus needs to narrow results by subject area, specific paper IDs, or authors. OracleVS supports metadata filters that combine with vector similarity.

### Filter by Subject Area

In [ ]:
query = "Find papers related to mission planning and observational astronomy"

docs = vector_store.similarity_search(
    query,
    k=3,
    filter={"primary_subject": {"$eq": sample_primary_subject}},
)

for doc in docs:
    print("Title:", doc.metadata.get("title", "N/A"))
    print("Subject:", doc.metadata.get("primary_subject", "N/A"))
    print("Text:", doc.page_content[:150], "...")
    print("------")


### Filter by Paper ID

In [ ]:
docs = vector_store.similarity_search(
    query="Explain key themes in this research paper",
    k=5,
    filter={"id": {"$in": [sample_arxiv_id]}},
)

for doc in docs:
    print("Title:", doc.metadata.get("title", "N/A"))
    print("ArXiv ID:", doc.metadata.get("arxiv_id", "N/A"))
    print("------")


--------

## Lab 2 Recap

You've now built the search foundation that Proteus will rely on:

| What You Did | Why It Matters |
|-------------|----------------|
| Created a vector-enabled SQL table | Documents stored with embeddings for semantic retrieval |
| Built an HNSW index | Fast approximate nearest-neighbor search at scale |
| Ingested 1,000 arXiv research papers | Real academic data with rich metadata (subject, authors, IDs) |
| Queried with natural language | "Planetary exploration" finds relevant papers without keyword matching |
| Applied metadata filters | Narrow results by subject area or specific paper IDs |

**Next up**: In Lab 3, we'll design the complete memory architecture that gives Proteus seven distinct types of memory — each with a specific purpose and storage strategy.


---

# End of Lab 2 — Continue to Lab 3

---

# Lab 3: Designing the Memory Architecture

## Memory Types, Design Decisions, and Storage Setup

--------

## Step 1: Define Table Names

Each memory type gets its own table. SQL tables for exact-match retrieval (conversational history, tool logs); vector-enabled SQL tables for semantic search (everything else).

In [ ]:
# Table names for each memory type
CONVERSATIONAL_TABLE   = "CONVERSATIONAL_MEMORY"   # Episodic memory
KNOWLEDGE_BASE_TABLE   = "SEMANTIC_MEMORY"          # Semantic memory
WORKFLOW_TABLE         = "WORKFLOW_MEMORY"           # Procedural memory
TOOLBOX_TABLE          = "TOOLBOX_MEMORY"            # Procedural memory
ENTITY_TABLE           = "ENTITY_MEMORY"             # Semantic memory
SUMMARY_TABLE          = "SUMMARY_MEMORY"            # Semantic memory
TOOL_LOG_TABLE         = "TOOL_LOG"                  # Episodic memory

ALL_TABLES = [
    CONVERSATIONAL_TABLE, KNOWLEDGE_BASE_TABLE, WORKFLOW_TABLE,
    TOOLBOX_TABLE, ENTITY_TABLE, SUMMARY_TABLE, TOOL_LOG_TABLE,
]

# Drop existing tables to start fresh
for table in ALL_TABLES:
    try:
        with vector_conn.cursor() as cur:
            cur.execute(f"DROP TABLE {table} PURGE")
    except Exception as e:
        if "ORA-00942" in str(e):
            print(f"  - {table} (not exists)")
        else:
            print(f"  ✗ {table}: {e}")

vector_conn.commit()

In [ ]:
# Model token limits (for context management in Lab 5)
MODEL_TOKEN_LIMITS = {
    "gpt-5": 256000,
    "gpt-5-mini": 128000,
    "gpt-4o": 128000,
    "gpt-4-turbo": 128000,
    "gpt-4": 8192,
    "gpt-3.5-turbo": 16385,
}



--------

## Step 2: Create the Conversational Memory Table

Unlike semantic memories backed by vector stores, conversational memory uses a traditional SQL table because we need **exact retrieval by thread ID** (not similarity search). Each research session gets its own `thread_id`.

The table includes a `summary_id` column — when older messages are summarized and compressed, they're marked (not deleted) with a reference to the summary that replaced them.

In [ ]:
def create_conversational_history_table(conn, table_name: str = "CONVERSATIONAL_MEMORY"):
    """
    Create a table to store conversational history.

    Columns:
    - id:         Unique message identifier
    - thread_id:  Groups messages by research session / conversation
    - role:       'user' or 'assistant'
    - content:    The message text
    - timestamp:  When the message was stored
    - metadata:   Optional JSON metadata
    - summary_id: Links to the summary that compressed this message (NULL if unsummarized)
    """
    with conn.cursor() as cur:
        try:
            cur.execute(f"DROP TABLE {table_name}")
        except:
            pass

        cur.execute(f"""
            CREATE TABLE {table_name} (
                id VARCHAR2(100) DEFAULT SYS_GUID() PRIMARY KEY,
                thread_id VARCHAR2(100) NOT NULL,
                role VARCHAR2(50) NOT NULL,
                content CLOB NOT NULL,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                metadata CLOB,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                summary_id VARCHAR2(100) DEFAULT NULL
            )
        """)

        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_thread_id ON {table_name}(thread_id)
        """)

        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_timestamp ON {table_name}(timestamp)
        """)

    conn.commit()
    print(f"✅ Table {table_name} created with indexes (thread_id, timestamp)")
    return table_name

In [ ]:
CONVERSATION_HISTORY_TABLE = create_conversational_history_table(
    vector_conn, CONVERSATIONAL_TABLE
)



--------

## Step 3: Create the Tool Log Table

Tool call outputs during agent execution can **bloat the context window** quickly — a single web search might return thousands of tokens that are only needed once.

The `TOOL_LOG` table acts as an **experimental memory**: full tool outputs are persisted to the database and replaced in the context window with a compact one-line reference. Proteus can retrieve full outputs later if needed.

This is a form of **context offloading** — keeping the working memory lean while preserving full fidelity in durable storage.

In [ ]:
def create_tool_log_table(conn, table_name: str = "TOOL_LOG"):
    """Create a table to log tool call outputs (experimental memory)."""
    with conn.cursor() as cur:
        try:
            cur.execute(f"DROP TABLE {table_name}")
        except:
            pass

        cur.execute(f"""
            CREATE TABLE {table_name} (
                id VARCHAR2(100) DEFAULT SYS_GUID() PRIMARY KEY,
                thread_id VARCHAR2(100) NOT NULL,
                tool_call_id VARCHAR2(200) NOT NULL,
                tool_name VARCHAR2(200) NOT NULL,
                tool_args CLOB,
                tool_output CLOB,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        cur.execute(
            f"CREATE INDEX idx_{table_name.lower()}_thread ON {table_name}(thread_id)"
        )
    conn.commit()
    print(f"✅ Table {table_name} created")
    return table_name


TOOL_LOG_TABLE_NAME = create_tool_log_table(vector_conn, TOOL_LOG_TABLE)


--------

## Step 4: Create Vector-Enabled Tables for Semantic Memories

Here we create five separate OracleVS-backed vector stores — one for each semantic memory type. Each uses the same embedding model for consistency.

| Vector Store Handle | Purpose |
|---------------------|---------|
| `knowledge_base_vs` | Research papers, web search results, curated references |
| `workflow_vs` | Learned search-and-analysis patterns (tool sequences that worked) |
| `toolbox_vs` | Tool definitions for semantic tool discovery |
| `entity_vs` | Extracted entities: paper titles, authors, research topics, arXiv IDs |
| `summary_vs` | Compressed summaries for long research sessions |

In [ ]:
knowledge_base_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=KNOWLEDGE_BASE_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

workflow_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=WORKFLOW_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

toolbox_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=TOOLBOX_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

entity_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=ENTITY_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

summary_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=SUMMARY_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)


### Build HNSW Indexes for Each Vector Store

In [ ]:
print("Creating vector indexes...")
safe_create_index(vector_conn, knowledge_base_vs, "knowledge_base_vs_hnsw")
safe_create_index(vector_conn, workflow_vs, "workflow_vs_hnsw")
safe_create_index(vector_conn, toolbox_vs, "toolbox_vs_hnsw")
safe_create_index(vector_conn, entity_vs, "entity_vs_hnsw")
safe_create_index(vector_conn, summary_vs, "summary_vs_hnsw")
print("✅ All indexes created!")


--------

## Step 5: Seed the Knowledge Base with Research Papers

We'll reuse the arXiv papers from Lab 2 to populate the knowledge base memory. In production, this would be a continuous ingestion pipeline from institutional repositories, journal APIs, or preprint servers.

In [ ]:
# Seed knowledge base memory with arXiv papers from Lab 2
if "sampled_papers" in globals() and sampled_papers:
    kb_texts = [
        f"Title: {p['title']}\nAbstract: {p['abstract']}" for p in sampled_papers
    ]
    kb_meta = [
        {
            "id": p["arxiv_id"],
            "arxiv_id": p["arxiv_id"],
            "title": p["title"],
            "primary_subject": p["primary_subject"],
            "authors": p["authors"],
            "source_type": "arxiv_papers",
        }
        for p in sampled_papers
    ]
    knowledge_base_vs.add_texts(kb_texts, kb_meta)
    print(f"✅ Seeded knowledge base memory with {len(kb_texts)} arXiv papers")


--------

## Lab 3 Recap

You've designed and created the complete memory infrastructure for Proteus:

| What You Did | Why It Matters |
|-------------|----------------|
| Defined 6 memory types with clear purposes | Each memory serves a distinct cognitive function |
| Chose SQL vs. vector storage per type | Exact retrieval (threads) vs. semantic search (meaning) |
| Designed programmatic vs. agent-triggered split | Reliability for core operations, flexibility for strategy |
| Created conversational memory table | Thread-based chat history with summary linkage |
| Created tool log table | Context offloading for lean working memory |
| Created 5 vector-enabled tables | Semantic search across knowledge, workflows, tools, entities, summaries |
| Seeded the knowledge base | arXiv research papers ready for Proteus to search |

**Key Insight**: The `summary_id` column in conversational memory enables **log compaction** — a pattern borrowed from databases where old entries are compressed but not lost. Messages are *marked* as summarized, not deleted, preserving full audit history.

**Next up**: In Lab 4, we'll implement the `MemoryManager` class that provides clean read/write interfaces for all these memory types, and build the semantic `Toolbox` for dynamic tool discovery.


---

# End of Lab 3 — Continue to Lab 4

---

# Lab 4: Building the Memory Manager & Toolbox

## The MemoryManager Class and Semantic Tool Discovery

--------

## Step 1: The MemoryManager Class

The `MemoryManager` is the central abstraction that unifies all memory operations. It provides a clean interface for reading and writing to different memory types, hiding the complexity of SQL queries and vector store operations.

### Key Features

- **Thread-based conversations** — Messages are organized by `thread_id` (one per research session)
- **Semantic search** — Vector stores enable finding relevant content by meaning
- **Metadata filtering** — Workflows filter by `num_steps > 0`, summaries filter by `id`
- **LLM-powered entity extraction** — Automatically extracts paper titles, authors, and research topics from text
- **Formatted context output** — Each read method returns formatted text ready for the LLM context window

### Alternative Frameworks

There are existing frameworks that abstract memory management for AI agents:

| Framework | Description |
|-----------|-------------|
| **LangChain Memory** | Built-in memory classes (ConversationBufferMemory, VectorStoreRetrieverMemory) |
| **Mem0** | Dedicated memory layer for AI agents with automatic memory management |
| **LlamaIndex** | Document-based memory with various storage backends |
| **Zep** | Long-term memory service for AI assistants |

For learning purposes, building your own memory manager (as we do here) gives you a deep understanding of how memory engineering works. For production, you might consider using or extending an existing framework. Note that this workshop only illustrates reads and writes — a production system would also need deletion, updates, and TTL-based expiry.

--------

### The Implementation

In [ ]:
import json as json_lib
from datetime import datetime


class MemoryManager:
    """
    Memory manager for AI agents using Oracle AI Database.

    Manages 7 types of memory:
    - Conversational: Chat history per research session (SQL table)
    - Knowledge Base: Searchable documents and KB articles (vector-enabled SQL table)
    - Workflow: Learned resolution patterns (vector-enabled SQL table)
    - Toolbox: Available diagnostic tools (vector-enabled SQL table)
    - Entity: Servers, services, people, teams (vector-enabled SQL table)
    - Summary: Compressed context from long sessions (vector-enabled SQL table)
    - Tool Log: Offloaded tool outputs for lean context (SQL table)
    """

    def __init__(
        self,
        conn,
        conversation_table: str,
        knowledge_base_vs,
        workflow_vs,
        toolbox_vs,
        entity_vs,
        summary_vs,
        tool_log_table: str = None,
    ):
        self.conn = conn
        self.conversation_table = conversation_table
        self.knowledge_base_vs = knowledge_base_vs
        self.workflow_vs = workflow_vs
        self.toolbox_vs = toolbox_vs
        self.entity_vs = entity_vs
        self.summary_vs = summary_vs
        self.tool_log_table = tool_log_table

    # ==================== CONVERSATIONAL MEMORY (SQL) ====================

    def write_conversational_memory(self, content: str, role: str, thread_id: str) -> str:
        """Store a message in conversation history."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            id_var = cur.var(str)
            cur.execute(
                f"""
                INSERT INTO {self.conversation_table}
                    (thread_id, role, content, metadata, timestamp)
                VALUES (:thread_id, :role, :content, :metadata, CURRENT_TIMESTAMP)
                RETURNING id INTO :id
            """,
                {
                    "thread_id": thread_id,
                    "role": role,
                    "content": content,
                    "metadata": "{}",
                    "id": id_var,
                },
            )
            record_id = id_var.getvalue()[0] if id_var.getvalue() else None
        self.conn.commit()
        return record_id

    def get_unsummarized_messages(self, thread_id: str, limit: int = 100) -> list[dict]:
        """Return unsummarized conversation turns for a thread."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT id, role, content, timestamp
                FROM {self.conversation_table}
                WHERE thread_id = :thread_id AND summary_id IS NULL
                ORDER BY timestamp ASC
                FETCH FIRST :limit ROWS ONLY
            """,
                {"thread_id": thread_id, "limit": limit},
            )
            rows = cur.fetchall()

        return [
            {"id": rid, "role": role, "content": content, "timestamp": ts}
            for rid, role, content, ts in rows
        ]

    def read_conversational_memory(self, thread_id: str, limit: int = 10) -> str:
        """Read unsummarized conversation history for a thread.

        NOTE: Only returns messages where summary_id IS NULL. Once messages
        are summarized via summarize_conversation(), they are excluded here
        and replaced by a compact summary reference in Summary Memory.
        """
        messages = self.get_unsummarized_messages(thread_id, limit=limit)
        lines = [
            f"[{m['timestamp'].strftime('%H:%M:%S')}] [{m['role']}] {m['content']}"
            for m in messages
        ]
        messages_formatted = "\n".join(lines)
        return f"""## Conversation Memory (Ticket: {thread_id})
### Purpose: Recent dialogue turns that have NOT yet been summarized.
### When to use: Refer to this for the user's latest questions, your prior answers,
### and any commitments or follow-ups from the current troubleshooting session.
### If context grows too long, call summarize_conversation(thread_id) to compact.

{messages_formatted}"""

    def mark_as_summarized(
        self, thread_id: str, summary_id: str, message_ids: list[str] | None = None
    ):
        """Mark conversation turns as summarized."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            if message_ids:
                cur.executemany(
                    f"""
                    UPDATE {self.conversation_table}
                    SET summary_id = :summary_id
                    WHERE thread_id = :thread_id AND id = :id AND summary_id IS NULL
                    """,
                    [
                        {"summary_id": summary_id, "thread_id": thread_id, "id": mid}
                        for mid in message_ids
                    ],
                )
                count = len(message_ids)
            else:
                cur.execute(
                    f"""
                    UPDATE {self.conversation_table}
                    SET summary_id = :summary_id
                    WHERE thread_id = :thread_id AND summary_id IS NULL
                """,
                    {"summary_id": summary_id, "thread_id": thread_id},
                )
                count = cur.rowcount
        self.conn.commit()
        print(f"  📦 Marked {count} messages as summarized (summary_id: {summary_id})")

    # ==================== KNOWLEDGE BASE (Vector-Enabled SQL Table) ====================

    def write_knowledge_base(self, text: str, metadata: dict):
        """Store text in knowledge base with metadata."""
        self.knowledge_base_vs.add_texts([text], [metadata])

    def read_knowledge_base(self, query: str, k: int = 3) -> str:
        """Search knowledge base for relevant content."""
        results = self.knowledge_base_vs.similarity_search(query, k=k)
        content = "\n".join([doc.page_content for doc in results])
        return f"""## Knowledge Base Memory
### Purpose: Research papers, web search results, and curated references stored for long-term reference.
### When to use: Cite specific facts, findings, or paper details from here before
### resorting to external search. If the KB lacks what you need, use search_tavily() to fetch
### new information (which will be stored here automatically).

{content}"""

    # ==================== WORKFLOW (Vector-Enabled SQL Table) ====================

    def write_workflow(self, query: str, steps: list, final_answer: str, success: bool = True):
        """Store a completed workflow pattern for future reference."""
        steps_text = "\n".join([f"Step {i+1}: {s}" for i, s in enumerate(steps)])
        text = f"Query: {query}\nSteps:\n{steps_text}\nAnswer: {final_answer[:200]}"

        metadata = {
            "query": query,
            "success": success,
            "num_steps": len(steps),
            "timestamp": datetime.now().isoformat(),
        }
        self.workflow_vs.add_texts([text], [metadata])

    def read_workflow(self, query: str, k: int = 3) -> str:
        """Search for similar past workflows with at least 1 step."""
        results = self.workflow_vs.similarity_search(
            query, k=k, filter={"num_steps": {"$gt": 0}}
        )
        if not results:
            return "## Workflow Memory\nNo relevant workflows found."
        content = "\n---\n".join([doc.page_content for doc in results])
        return f"""## Workflow Memory
### Purpose: Step-by-step records of how similar past research queries were resolved.
### When to use: Before planning a multi-step resolution, check if a similar workflow
### already succeeded. Reuse proven tool sequences instead of re-discovering them.

{content}"""

    # ==================== TOOLBOX (Vector-Enabled SQL Table) ====================

    def write_toolbox(self, text: str, metadata: dict):
        """Store a tool definition in the toolbox."""
        self.toolbox_vs.add_texts([text], [metadata])

    def read_toolbox(self, query: str, k: int = 3) -> list[dict]:
        """Find relevant tools and return OpenAI-compatible schemas."""
        results = self.toolbox_vs.similarity_search(query, k=k)
        tools = []
        for doc in results:
            meta = doc.metadata
            stored_params = meta.get("parameters", {})
            properties = {}
            required = []

            for param_name, param_info in stored_params.items():
                param_type = param_info.get("type", "string")
                type_mapping = {
                    "<class 'str'>": "string",
                    "<class 'int'>": "integer",
                    "<class 'float'>": "number",
                    "<class 'bool'>": "boolean",
                    "str": "string",
                    "int": "integer",
                    "float": "number",
                    "bool": "boolean",
                }
                json_type = type_mapping.get(param_type, "string")
                properties[param_name] = {"type": json_type}

                if "default" not in param_info:
                    required.append(param_name)

            tools.append(
                {
                    "type": "function",
                    "function": {
                        "name": meta.get("name", "tool"),
                        "description": meta.get("description", ""),
                        "parameters": {
                            "type": "object",
                            "properties": properties,
                            "required": required,
                        },
                    },
                }
            )
        return tools

    # ==================== ENTITY (Vector-Enabled SQL Table) ====================

    def extract_entities(self, text: str, llm_client) -> list[dict]:
        """Use LLM to extract entities (paper titles, authors, research topics) from text."""
        if not text or len(text.strip()) < 5:
            return []

        prompt = f'''Extract entities from: "{text[:500]}"
Return JSON: [{{"name": "X", "type": "PAPER|AUTHOR|TOPIC|INSTITUTION|METHOD", "description": "brief"}}]
If none: []'''

        try:
            response = llm_client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=300,
            )
            result = response.choices[0].message.content.strip()

            start, end = result.find("["), result.rfind("]")
            if start == -1 or end == -1:
                return []

            parsed = json_lib.loads(result[start : end + 1])
            return [
                {
                    "name": e["name"],
                    "type": e.get("type", "UNKNOWN"),
                    "description": e.get("description", ""),
                }
                for e in parsed
                if isinstance(e, dict) and e.get("name")
            ]
        except:
            return []

    def write_entity(
        self, name: str, entity_type: str, description: str, llm_client=None, text: str = None
    ):
        """Store an entity OR extract and store entities from text."""
        if text and llm_client:
            entities = self.extract_entities(text, llm_client)
            for e in entities:
                self.entity_vs.add_texts(
                    [f"{e['name']} ({e['type']}): {e['description']}"],
                    [{"name": e["name"], "type": e["type"], "description": e["description"]}],
                )
            return entities
        else:
            self.entity_vs.add_texts(
                [f"{name} ({entity_type}): {description}"],
                [{"name": name, "type": entity_type, "description": description}],
            )

    def read_entity(self, query: str, k: int = 5) -> str:
        """Search for relevant entities."""
        results = self.entity_vs.similarity_search(query, k=k)
        if not results:
            return "## Entity Memory\nNo entities found."

        entities = [
            f"• {doc.metadata.get('name', '?')}: {doc.metadata.get('description', '')}"
            for doc in results
            if hasattr(doc, "metadata")
        ]
        entities_formatted = "\n".join(entities)
        return f"""## Entity Memory
### Purpose: Named entities (paper titles, authors, research topics, methods) extracted from conversations.
### When to use: Resolve references like "that paper" or "the author we discussed".
### Entity memory provides continuity — ground your answers in known entities
### rather than guessing or re-asking for names already mentioned.

{entities_formatted}"""

    # ==================== SUMMARY (Vector-Enabled SQL Table) ====================

    def write_summary(self, summary_id: str, full_content: str, summary: str, description: str):
        """Store a summary with its original content."""
        self.summary_vs.add_texts(
            [f"{summary_id}: {description}"],
            [
                {
                    "id": summary_id,
                    "full_content": full_content,
                    "summary": summary,
                    "description": description,
                }
            ],
        )
        return summary_id

    def read_summary_memory(self, summary_id: str) -> str:
        """Retrieve a specific summary by ID (just-in-time retrieval)."""
        results = self.summary_vs.similarity_search(
            summary_id, k=5, filter={"id": summary_id}
        )
        if not results:
            return f"Summary {summary_id} not found."
        doc = results[0]
        return doc.metadata.get("summary", "No summary content.")

    def read_summary_context(self, query: str = "", k: int = 10) -> str:
        """Get available summaries for context window (IDs + descriptions only)."""
        results = self.summary_vs.similarity_search(query or "summary", k=k)
        if not results:
            return "## Summary Memory\nNo summaries available."

        lines = [
            "## Summary Memory",
            "### Purpose: Compressed snapshots of older troubleshooting sessions.",
            "### When to use: These are lightweight pointers. If a summary looks relevant,",
            "### call expand_summary(summary_id) to retrieve the full content just-in-time.",
            "### Do NOT expand all summaries — only expand when you need specific details.",
            "",
        ]
        for doc in results:
            sid = doc.metadata.get("id", "?")
            desc = doc.metadata.get("description", "No description")
            lines.append(f"  • [ID: {sid}] {desc}")
        return "\n".join(lines)

    # ==================== TOOL LOG (SQL - Experimental Memory) ====================

    def write_tool_log(
        self, thread_id: str, tool_call_id: str, tool_name: str, tool_args: str, tool_output: str
    ) -> str:
        """Log a tool call output to the database and return a compact reference."""
        if not self.tool_log_table:
            return tool_output

        with self.conn.cursor() as cur:
            id_var = cur.var(str)
            cur.execute(
                f"""
                INSERT INTO {self.tool_log_table}
                    (thread_id, tool_call_id, tool_name, tool_args, tool_output)
                VALUES (:thread_id, :tool_call_id, :tool_name, :tool_args, :tool_output)
                RETURNING id INTO :id
            """,
                {
                    "thread_id": str(thread_id),
                    "tool_call_id": tool_call_id,
                    "tool_name": tool_name,
                    "tool_args": tool_args,
                    "tool_output": tool_output,
                    "id": id_var,
                },
            )
            log_id = id_var.getvalue()[0] if id_var.getvalue() else None
        self.conn.commit()

        preview = tool_output[:150].replace("\n", " ")
        return f"[Tool Log {log_id}] {tool_name} executed. Preview: {preview}..."

    def read_tool_log(self, thread_id: str, limit: int = 20) -> list[dict]:
        """Read tool call logs for a thread."""
        if not self.tool_log_table:
            return []
        with self.conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT id, tool_call_id, tool_name, tool_args, tool_output, timestamp
                FROM {self.tool_log_table}
                WHERE thread_id = :thread_id
                ORDER BY timestamp DESC
                FETCH FIRST :limit ROWS ONLY
            """,
                {"thread_id": str(thread_id), "limit": limit},
            )
            rows = cur.fetchall()
        return [
            {
                "id": r[0], "tool_call_id": r[1], "tool_name": r[2],
                "tool_args": r[3], "tool_output": r[4], "timestamp": r[5],
            }
            for r in rows
        ]

    # ==================== SUMMARY EXPANSION HELPERS ====================

    def get_messages_by_summary_id(self, summary_id: str) -> list[dict]:
        """Retrieve original messages that were compacted into a given summary."""
        with self.conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT id, role, content, timestamp
                FROM {self.conversation_table}
                WHERE summary_id = :summary_id
                ORDER BY timestamp ASC
            """,
                {"summary_id": summary_id},
            )
            rows = cur.fetchall()
        return [
            {"id": rid, "role": role, "content": content, "timestamp": ts}
            for rid, role, content, ts in rows
        ]


--------

## Step 2: Initialize the Memory Manager

In [ ]:
memory_manager = MemoryManager(
    conn=vector_conn,
    conversation_table=CONVERSATION_HISTORY_TABLE,
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs,
    tool_log_table=TOOL_LOG_TABLE_NAME,
)


### Quick Smoke Test

Let's verify the memory manager works by writing and reading a test conversation:

In [ ]:
# Write a test conversation
test_thread = "SESSION-TEST-001"
memory_manager.write_conversational_memory("Can you find papers on transformer architectures for time-series forecasting?", "user", test_thread)
memory_manager.write_conversational_memory("Let me search the knowledge base for relevant papers.", "assistant", test_thread)

# Read it back
print(memory_manager.read_conversational_memory(test_thread))

In [ ]:
# Test knowledge base search
print(memory_manager.read_knowledge_base("transformer architectures for time-series"))



--------

## Step 3: The Semantic Toolbox

### The Scalability Problem with Tools

As your AI system grows, you might have **hundreds of tools** available — diagnostic scripts, API calls, database queries, search endpoints. Passing all tools to the LLM at inference time creates serious problems:

| Problem | Impact |
|---------|--------|
| **Context bloat** | Tool definitions consume tokens, leaving less room for actual content |
| **Tool selection failure** | LLMs struggle to choose the right tool when presented with too many options |
| **Increased latency** | More tokens = slower inference |
| **Higher costs** | More tokens = higher API costs |

Model providers typically recommend limiting tools to 10-20 max for reliable selection.

### The Solution: Semantic Tool Retrieval

The `Toolbox` class solves this by treating tools as a **searchable memory**:

1. **Register hundreds of tools** — Store all available tools with embeddings of their descriptions
2. **Retrieve only relevant tools** — At inference time, vector search finds tools semantically relevant to the current query
3. **Pass a focused toolset** — Only the top 3-5 retrieved tools are passed to the LLM

This means your system can **scale to hundreds of tools** while the LLM only sees the most relevant ones for each query.

### How It Works

```
User Query → Embed Query → Vector Search → Find tools with similar docstrings → Return relevant tools
```

The `augment=True` flag triggers LLM-powered enhancement:

1. **Docstring augmentation** — LLM rewrites the docstring to be clearer and more searchable
2. **Synthetic query generation** — LLM generates example queries that would need this tool
3. **Rich embedding** — Combines name + augmented docstring + signature + queries for better retrieval

This means a simple docstring like `"Search the web"` becomes a rich description that matches when the user asks `"What are the latest advances in diffusion models?"`.

### Three Engineering Disciplines in One

| Discipline | Technique Used | How It Helps |
|------------|----------------|--------------|
| **Memory Engineering** | Toolbox as procedural memory | Tools are stored and retrieved like learned skills |
| **Memory Engineering** | Docstring augmentation | LLM improves descriptions for better retrieval |
| **Context Engineering** | Selective tool retrieval | Only relevant tools enter the context window |
| **Prompt Engineering** | Role setting | "You are a technical writer" improves docstring quality |

--------

### The Implementation

In [ ]:
import inspect
import uuid
from typing import Callable, Optional, Union
from pydantic import BaseModel


def get_embedding(text: str) -> list[float]:
    """Get the embedding for a text using the configured embedding model."""
    return embedding_model.embed_query(text)


class ToolMetadata(BaseModel):
    """Metadata for a registered tool."""
    name: str
    description: str
    signature: str
    parameters: dict
    return_type: str


class Toolbox:
    """
    Toolbox for registering, storing, and retrieving tools with LLM-powered augmentation.

    Tools are stored with embeddings for semantic retrieval, allowing Proteus to
    find relevant research tools based on natural language queries.
    """

    def __init__(self, memory_manager, llm_client, model: str = "gpt-4o"):
        self.memory_manager = memory_manager
        self.llm_client = llm_client
        self.model = model
        self._tools: dict[str, Callable] = {}
        self._tools_by_name: dict[str, Callable] = {}

    def _augment_docstring(self, docstring: str) -> str:
        """Use LLM to improve and expand a tool's docstring for better retrieval."""
        if not docstring.strip():
            return "No description provided."

        prompt = f"""You are a technical writer. Improve the following function docstring to be more clear,
            comprehensive, and useful. Include:
            1. A clear concise summary
            2. Detailed description of what the function does
            3. When to use this function
            4. Any important notes or caveats

            Original docstring:
            {docstring}

            Return ONLY the improved docstring, no other text.
        """

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=500,
        )
        return response.choices[0].message.content.strip()

    def _generate_queries(self, docstring: str, num_queries: int = 5) -> list[str]:
        """Generate synthetic example queries that would lead to using this tool."""
        prompt = f"""Based on the following tool description, generate {num_queries} diverse example queries
            that a user might ask when they need this tool. Make them natural and varied.

            Tool description:
            {docstring}

            Return ONLY a JSON array of strings, like: ["query1", "query2", ...]
        """

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=300,
        )

        try:
            import json
            queries = json.loads(response.choices[0].message.content.strip())
            return queries if isinstance(queries, list) else []
        except json.JSONDecodeError:
            return [response.choices[0].message.content.strip()]

    def _get_tool_metadata(self, func: Callable) -> ToolMetadata:
        """Extract metadata from a function for storage and retrieval."""
        sig = inspect.signature(func)

        parameters = {}
        for name, param in sig.parameters.items():
            param_info = {"name": name}
            if param.annotation != inspect.Parameter.empty:
                param_info["type"] = str(param.annotation)
            if param.default != inspect.Parameter.empty:
                param_info["default"] = str(param.default)
            parameters[name] = param_info

        return_type = "Any"
        if sig.return_annotation != inspect.Signature.empty:
            return_type = str(sig.return_annotation)

        return ToolMetadata(
            name=func.__name__,
            description=func.__doc__ or "No description",
            signature=str(sig),
            parameters=parameters,
            return_type=return_type,
        )

    def register_tool(
        self, func: Optional[Callable] = None, augment: bool = False
    ) -> Union[str, Callable]:
        """
        Register a function as a tool in the toolbox.

        Can be used as a decorator or called directly:

            @toolbox.register_tool
            def my_tool(): ...

            @toolbox.register_tool(augment=True)
            def my_enhanced_tool(): ...
        """

        def decorator(f: Callable) -> str:
            docstring = f.__doc__ or ""
            signature = str(inspect.signature(f))
            object_id = uuid.uuid4()
            object_id_str = str(object_id)

            if augment:
                augmented_docstring = self._augment_docstring(docstring)
                queries = self._generate_queries(augmented_docstring)

                embedding_text = (
                    f"{f.__name__} {augmented_docstring} {signature} {' '.join(queries)}"
                )
                embedding = get_embedding(embedding_text)

                tool_data = self._get_tool_metadata(f)
                tool_data.description = augmented_docstring

                tool_dict = {
                    "_id": object_id_str,
                    "embedding": embedding,
                    "queries": queries,
                    "augmented": True,
                    **tool_data.model_dump(),
                }
            else:
                embedding = get_embedding(f"{f.__name__} {docstring} {signature}")
                tool_data = self._get_tool_metadata(f)

                tool_dict = {
                    "_id": object_id_str,
                    "embedding": embedding,
                    "augmented": False,
                    **tool_data.model_dump(),
                }

            self.memory_manager.write_toolbox(
                f"{f.__name__} {docstring} {signature}", tool_dict
            )

            self._tools[object_id_str] = f
            self._tools_by_name[f.__name__] = f
            return object_id_str

        if func is None:
            return decorator
        return decorator(func)


--------

## Step 4: Initialize the Toolbox and Set Up API Keys

In [ ]:
import os
import getpass


def set_env_securely(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

In [ ]:
set_env_securely("OPENAI_API_KEY", "OpenAI API Key: ")

In [ ]:
from openai import OpenAI

client = OpenAI()

# Initialize the Toolbox
toolbox = Toolbox(memory_manager=memory_manager, llm_client=client)




--------

## Step 5: Test Tool Registration and Retrieval

Let's register a simple diagnostic tool and verify semantic retrieval works:

In [ ]:
@toolbox.register_tool(augment=True)
def lookup_paper_details(arxiv_id: str) -> str:
    """Look up detailed metadata for a research paper by its arXiv ID."""
    # In production, this would call the arXiv API
    mock_papers = {
        "1706.03762": "Title: Attention Is All You Need | Authors: Vaswani et al. | Subject: cs.CL | Year: 2017",
        "2005.14165": "Title: Language Models are Few-Shot Learners | Authors: Brown et al. | Subject: cs.CL | Year: 2020",
        "2303.08774": "Title: GPT-4 Technical Report | Authors: OpenAI | Subject: cs.CL | Year: 2023",
    }
    return mock_papers.get(
        arxiv_id, f"❓ Paper not found: {arxiv_id}"
    )

In [ ]:
# Test semantic retrieval — does the toolbox find this tool for a related query?
import pprint

retrieved_tools = memory_manager.read_toolbox("look up details for a specific research paper")
pprint.pprint(retrieved_tools)



> **🔍 Try it**: Change the query to something different — "find the authors of this paper" or "get metadata for arXiv 1706.03762" — and see if the tool is still retrieved. This is semantic retrieval in action.

--------

## Lab 4 Recap

| What You Built | Why It Matters |
|---------------|----------------|
| `MemoryManager` class with 7 memory types | Unified read/write interface hiding SQL and vector complexity |
| Thread-based conversation read/write | Multi-session support with independent history per research thread |
| Summary marking (not deletion) | Log compaction pattern — compress without losing audit trail |
| Entity extraction via LLM | Automatic recognition of papers, authors, and topics from conversation |
| `Toolbox` class with semantic retrieval | Scale to hundreds of tools while LLM only sees relevant ones |
| LLM-augmented tool registration | Better retrieval through enhanced descriptions and synthetic queries |

**Key Insight**: The Toolbox sits at the intersection of three disciplines: *memory engineering* (tools as procedural memory), *context engineering* (only relevant tools in context), and *prompt engineering* (role-setting for better docstring augmentation).

**Next up**: In Lab 5, we'll build the context engineering layer — usage tracking, summarization, just-in-time retrieval — and integrate Tavily for web search.


---

# End of Lab 4 — Continue to Lab 5

---

# Lab 5: Context Engineering & Web Integration

## Context Window Management, Summarization, and Tavily Search

--------

## Step 1: Context Window Usage Calculator

This simple utility estimates how much of the context window is being used. Proteus can check this to decide whether compaction is needed.

In [ ]:
def calculate_context_usage(context: str, model: str = "gpt-4o") -> dict:
    """Calculate context window usage as percentage."""
    estimated_tokens = len(context) // 4  # ~4 chars per token
    max_tokens = MODEL_TOKEN_LIMITS.get(model, 128000)
    percentage = (estimated_tokens / max_tokens) * 100
    return {
        "tokens": estimated_tokens,
        "max": max_tokens,
        "percent": round(percentage, 1),
    }


--------

## Step 2: Context Summarizer

When the context window grows large — after several tool calls, long conversations, or large search results — we can compress it into a summary. The full content is stored in Summary Memory, and the context window gets a compact pointer.

In [ ]:
import uuid


def summarise_context_window(
    content: str, memory_manager, llm_client, model: str = "gpt-4o"
) -> dict:
    """Summarise context window using LLM and store in summary memory."""
    summary_prompt = f"""
You are compressing an AI research assistant's context window for later retrieval.
The content may include conversation memory, KB articles, entities, workflows, and prior summaries.

Produce a compact summary that preserves:
- user's reported issue and constraints
- key diagnostic findings already established
- important entities (paper titles, author names, arXiv IDs, research topics)
- unresolved questions and next actions

Output 4-7 short bullet points.
Be faithful to the source, and do not add new facts.

Context window content:
{content[:3000]}
""".strip()

    response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": summary_prompt}],
        max_completion_tokens=220,
    )
    summary = response.choices[0].message.content

    desc_response = llm_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": f"Write a short label (max 12 words) for this summary:\n{summary}",
            }
        ],
        max_completion_tokens=40,
    )
    description = desc_response.choices[0].message.content.strip()

    summary_id = str(uuid.uuid4())[:8]
    memory_manager.write_summary(summary_id, content, summary, description)

    return {"id": summary_id, "description": description, "summary": summary}


--------

## Step 3: Context Offloader

This utility checks whether the context exceeds a threshold and, if so, automatically summarizes and replaces the content with a compact reference.

In [ ]:
def offload_to_summary(
    context: str,
    memory_manager,
    llm_client,
    threshold_percent: float = 80.0,
) -> tuple:
    """If context exceeds threshold, summarise and return compacted version."""
    usage = calculate_context_usage(context)

    if usage["percent"] < threshold_percent:
        return context, []  # No offload needed

    result = summarise_context_window(context, memory_manager, llm_client)

    compact = f"[Summary ID: {result['id']}] {result['description']}"
    return compact, [result]


--------

## Step 4: Register Summary Tools for the Agent

These are **agent-triggered** tools — Proteus decides when to call them based on the current context. We register them with `augment=True` for better semantic retrieval.

### Design Decision: Mark Instead of Delete

When conversation history grows large, we need to reduce context. We chose to **mark messages as summarized** rather than delete them:

| Approach | Pros | Cons |
|----------|------|------|
| **Delete summarized messages** | Simple, immediate space savings | Permanent data loss, can't audit or recover |
| **Mark as summarized (our choice)** | Preserves history, reversible, auditable | Slightly more complex queries |

Memory should be *compressed* or *forgotten*, not *erased*. The original messages remain for auditing, debugging, or reprocessing.

### The Compaction Flow

```
Ticket thread has 50 messages → Context too large → summarize_conversation(thread_id)
                                                          ↓
                               1. Read unsummarized messages
                               2. LLM summarizes them
                               3. Store summary with unique ID
                               4. UPDATE messages SET summary_id = 'abc123'
                                                          ↓
                               Next read: Only new messages appear + Summary ID reference
```

In [ ]:
@toolbox.register_tool(augment=True)
def expand_summary(summary_id: str) -> str:
    """Expand a summary reference to full content, including the original conversation
    messages that were compacted into it. Use when you need more details from a
    [Summary ID: xxx] reference during troubleshooting."""
    summary_text = memory_manager.read_summary_memory(summary_id)

    original_msgs = memory_manager.get_messages_by_summary_id(summary_id)
    if original_msgs:
        lines = [f"[{m['role']}] {m['content']}" for m in original_msgs]
        return (
            f"Summary:\n{summary_text}\n\n"
            f"Original messages ({len(original_msgs)}):\n" + "\n".join(lines)
        )
    return summary_text


@toolbox.register_tool(augment=True)
def summarize_and_store(text: str) -> str:
    """Summarize a long text block and store it. Returns [Summary ID: ...] for later expansion."""
    result = summarise_context_window(text, memory_manager, client)
    return f"Stored as [Summary ID: {result['id']}] {result['description']}"


@toolbox.register_tool(augment=True)
def summarize_conversation(thread_id: str) -> str:
    """
    Summarize unsummarized conversation turns for a research session thread and mark those
    turns with a summary_id. Use this when conversation memory becomes long and
    you need context compaction during a troubleshooting session.
    """
    unsummarized = memory_manager.get_unsummarized_messages(thread_id, limit=200)
    if not unsummarized:
        return "No unsummarized conversation turns found."

    full_text = "\n".join([f"[{m['role']}] {m['content']}" for m in unsummarized])
    result = summarise_context_window(full_text, memory_manager, client)

    message_ids = [m["id"] for m in unsummarized]
    memory_manager.mark_as_summarized(thread_id, result["id"], message_ids=message_ids)

    return f"Conversation summarized as [Summary ID: {result['id']}] {result['description']}"


--------

## Step 5: Web Search with Tavily

Proteus needs to search external sources when the internal knowledge base doesn't have the answer — for example, looking up a recent paper not yet in the knowledge base, a new research finding, or an unfamiliar technique.

We use [Tavily](https://tavily.com/), an AI-optimized search API designed for LLM applications.

### The Search-and-Store Pattern

When Proteus calls `search_tavily()`, it doesn't just return results — it **persists them to the knowledge base**:

```
Proteus calls search_tavily("recent advances in diffusion models 2026")
       ↓
Tavily API returns results
       ↓
Each result is written to knowledge_base_vs with metadata (title, URL, timestamp)
       ↓
Future sessions can retrieve this information without searching again
```

This pattern means Proteus **learns** from its searches. Information discovered once becomes part of the agent's long-term memory.

In [ ]:
set_env_securely("TAVILY_API_KEY", "Tavily API Key: ")

In [ ]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@toolbox.register_tool(augment=True)
def search_tavily(query: str, max_results: int = 5):
    """
    Search the web for information not available in the internal knowledge base.
    Results are automatically stored in the knowledge base for future reference.
    Use for vendor advisories, CVEs, error messages, and external documentation.
    """
    response = tavily_client.search(query=query, max_results=max_results)
    results = response.get("results", [])

    for result in results:
        text = (
            f"Title: {result.get('title', '')}\n"
            f"Content: {result.get('content', '')}\n"
            f"URL: {result.get('url', '')}"
        )

        metadata = {
            "title": result.get("title", ""),
            "url": result.get("url", ""),
            "score": result.get("score", 0),
            "source_type": "tavily_search",
            "query": query,
            "timestamp": datetime.now().isoformat(),
        }

        memory_manager.write_knowledge_base(text, metadata)

    return results



### Verify Tool Retrieval

Let's confirm that Proteus can find the search tool when needed:

In [ ]:
import pprint

retrieved_tools = memory_manager.read_toolbox("Search the internet for vendor documentation")
pprint.pprint(retrieved_tools)


--------

## Lab 5 Recap

| What You Built | Why It Matters |
|---------------|----------------|
| `calculate_context_usage()` | Monitor context consumption and trigger compaction proactively |
| `summarise_context_window()` | LLM-powered compression that preserves key diagnostic details |
| `offload_to_summary()` | Automatic threshold-based context offloading |
| `expand_summary()` tool | JIT retrieval — Proteus expands only the summaries it needs |
| `summarize_conversation()` tool | Log compaction for long troubleshooting threads |
| `search_tavily()` tool | External search with automatic knowledge base persistence |

**Key Insight**: The search-and-store pattern means Proteus builds institutional knowledge over time. The first time a SeerGroup employee asks about a specific error, Proteus searches externally. The second time, Proteus finds the answer in its own knowledge base — no external call needed, faster and cheaper.

**Next up**: In Lab 6, we'll wire everything together into the `call_agent` harness, run Proteus through real research scenarios, and compare the engineered approach against a naive baseline.


---

# End of Lab 5 — Continue to Lab 6

---

# Lab 6: Agent Execution & Evaluation

## Running Proteus Through Research Scenarios

--------

## Step 1: The Agent System Prompt

The system prompt tells Proteus how to use its memory systems and tools. Notice it establishes a **priority order** for memory types — this is critical for reliable behavior.

In [ ]:
import json as json_lib
from openai import OpenAI

client = OpenAI()

# Persistent context-window tracker — survives across call_agent() invocations
context_size_history = []  # list of (run_label, iteration, estimated_tokens)


AGENT_SYSTEM_PROMPT = """
# System Instructions
You are Proteus, SeerGroup's AI Research Assistant. You have access to memory systems and
research tools to help analysts navigate academic literature, synthesize findings, and track
research threads across sessions.

IMPORTANT: The user's input contains CONTEXT retrieved from multiple memory systems.
Each memory section has a Purpose and When-to-use guide — follow them.

## Memory Priority Order
1. **Conversation Memory** — check what the user already asked and what you already answered.
2. **Knowledge Base Memory** — cite facts from stored papers, abstracts, and web search results
   before searching externally.
3. **Entity Memory** — resolve named references ("that author", "the paper we discussed") from here.
4. **Workflow Memory** — reuse proven search-and-analysis sequences for similar past queries.
5. **Summary Memory** — expand a summary ID only when you need specific details from an older session.

## Tool Output Handling
Tool call outputs are logged to a Tool Log table and replaced with compact references in context.
The preview in each [Tool Log ...] reference contains enough to reason about the result.
If you need the full output, it can be retrieved from the database — but prefer working with
the preview and the knowledge base (where search results are also stored).

## Context Management
If conversation memory is getting long or repetitive, call summarize_conversation(thread_id)
to compact it. Use summarization tools at your discretion when they improve context quality.

When answering:
1. FIRST, use the context provided in the input
2. Expand summary IDs just-in-time when needed
3. Use external search tools only if memory context is insufficient
4. Keep responses evidence-based and grounded in the retrieved research context
5. Cite paper titles, authors, and arXiv IDs when available
"""


--------

## Step 2: Tool Execution and OpenAI Chat Wrapper

In [ ]:
def execute_tool(tool_name: str, tool_args: dict) -> str:
    """Execute a tool by looking it up in the toolbox."""
    if tool_name not in toolbox._tools_by_name:
        return f"Error: Tool '{tool_name}' not found"
    return str(toolbox._tools_by_name[tool_name](**tool_args) or "Done")


def call_openai_chat(messages: list, tools: list = None, model: str = "gpt-4o"):
    """Call OpenAI Chat Completions API with tools."""
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    return client.chat.completions.create(**kwargs)


--------

## Step 3: The Turn-Level Agent Harness

This is the core of Proteus. Each call to `call_agent()` represents one **agent run** (one user turn handled). Within a run, the **tool-call loop** repeats: model reasoning → optional tool calls → harness executes tools → model observes results → repeat until a final answer.

### The Flow

```
1. BUILD CONTEXT (programmatic)
   ├── Read conversational memory (unsummarized turns only)
   ├── Read knowledge base (relevant research papers)
   ├── Read workflow memory (past search-and-analysis patterns)
   ├── Read entity memory (papers, authors, topics)
   └── Read summary context (available summary IDs + descriptions)

2. GET TOOLS (programmatic)
   └── Retrieve semantically relevant tools from toolbox

3. STORE USER MESSAGE (programmatic)
   └── Persist the user message + best-effort entity extraction

4. WITHIN-RUN TOOL-CALL LOOP (up to max_iterations, within time budget)
   ├── Call LLM with context + tool schemas
   ├── If tool calls → execute tools and append results
   ├── If tools changed memory (search/compaction) → rebuild context
   └── If no tool calls → finalize answer

5. GUARDED STOP
   └── If budget hit → force a final best-effort answer (no tools)

6. SAVE RESULTS (programmatic)
   ├── Write workflow (if tools were used)
   ├── Best-effort entity extraction on final answer
   └── Store assistant response in conversational memory
```

In [ ]:
import time


def call_agent(
    query: str,
    thread_id: str = "1",
    max_iterations: int = 10,
    max_execution_time_s: float = 60.0,
) -> str:
    """Turn-level agent harness: build context, run tool-call loop, persist results.

    Appends (run_label, iteration, tokens) to the global context_size_history list
    so context growth can be visualised across multiple runs.
    """
    thread_id = str(thread_id)
    steps = []
    run_label = f"Run {len(set(r for r, _, _ in context_size_history)) + 1}"

    start_time = time.time()
    timed_out = False

    # ── 1. Build context from memory ──
    print("\n" + "=" * 50)
    print("🧠 BUILDING CONTEXT...")

    def build_context() -> str:
        """Rebuild the full context from the current memory state."""
        ctx = f"# Support Ticket\n{query}\n\n"
        ctx += memory_manager.read_conversational_memory(thread_id) + "\n\n"
        ctx += memory_manager.read_knowledge_base(query) + "\n\n"
        ctx += memory_manager.read_workflow(query) + "\n\n"
        ctx += memory_manager.read_entity(query) + "\n\n"
        ctx += memory_manager.read_summary_context(query) + "\n\n"
        return ctx

    context = build_context()

    print("==== CONTEXT WINDOW ====\n")
    print(context)

    # ── 2. Check context usage ──
    usage = calculate_context_usage(context)
    print(f"📊 Context: {usage['percent']}% ({usage['tokens']}/{usage['max']} tokens)")
    if usage["percent"] > 80:
        print(
            "⚠️ Context >80% — Proteus may call summarize_conversation(thread_id) for compaction."
        )

    # ── 3. Get tools ──
    dynamic_tools = memory_manager.read_toolbox(query, k=5)

    # Ensure summary tools are always available for discretionary compaction/JIT
    summary_tool_candidates = memory_manager.read_toolbox(
        "summarize conversation compact context expand summary memory", k=5
    )
    must_have = {"expand_summary", "summarize_conversation", "summarize_and_store"}
    existing = {t.get("function", {}).get("name") for t in dynamic_tools}

    for tool in summary_tool_candidates:
        name = tool.get("function", {}).get("name")
        if name in must_have and name not in existing:
            dynamic_tools.append(tool)
            existing.add(name)

    print(f"🔧 Tools: {[t['function']['name'] for t in dynamic_tools]}")

    # ── 4. Store user message & extract entities ──
    memory_manager.write_conversational_memory(query, "user", thread_id)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=query)
    except:
        pass

    # ── 5. Within-run tool-call loop ──
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": context},
    ]
    final_answer = ""

    tool_schema_tokens = len(json_lib.dumps(dynamic_tools)) // 4 if dynamic_tools else 0

    print("\n🤖 TOOL-CALL LOOP")
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")

        # Track context window size
        total_chars = sum(len(m.get("content", "") or "") for m in messages)
        est_tokens = (total_chars // 4) + tool_schema_tokens
        context_size_history.append((run_label, iteration + 1, est_tokens))

        if max_execution_time_s is not None:
            elapsed = time.time() - start_time
            if elapsed > max_execution_time_s:
                timed_out = True
                print(
                    f"\n⏱️ Time limit reached ({elapsed:.1f}s > {max_execution_time_s:.1f}s). Finalizing..."
                )
                break

        response = call_openai_chat(messages, tools=dynamic_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": msg.content,
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": "function",
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments,
                            },
                        }
                        for tc in msg.tool_calls
                    ],
                }
            )

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                raw_args = tc.function.arguments or "{}"
                try:
                    tool_args = json_lib.loads(raw_args)
                except Exception as e:
                    result = f"Error: invalid JSON arguments for {tool_name}: {e}"
                    print(f"🛠️ {tool_name}(<invalid args>)")
                    steps.append(f"{tool_name}(<invalid args>) → failed")
                    messages.append(
                        {"role": "tool", "tool_call_id": tc.id, "content": result}
                    )
                    continue

                if not isinstance(tool_args, dict):
                    result = f"Error: arguments for {tool_name} must be a JSON object."
                    print(f"🛠️ {tool_name}(<non-object args>)")
                    steps.append(f"{tool_name}(<non-object args>) → failed")
                    messages.append(
                        {"role": "tool", "tool_call_id": tc.id, "content": result}
                    )
                    continue

                # Ensure conversation compaction targets the active session thread
                if tool_name == "summarize_conversation":
                    tool_args["thread_id"] = thread_id

                args_display = {
                    k: (v[:50] + "..." if isinstance(v, str) and len(v) > 50 else v)
                    for k, v in tool_args.items()
                }
                print(f"🛠️ {tool_name}({args_display})")

                if max_execution_time_s is not None:
                    elapsed = time.time() - start_time
                    if elapsed > max_execution_time_s:
                        timed_out = True
                        result = f"Error: time limit reached before executing {tool_name}."
                        steps.append(f"{tool_name}({args_display}) → timed out")
                        messages.append(
                            {"role": "tool", "tool_call_id": tc.id, "content": result}
                        )
                        break

                try:
                    result = execute_tool(tool_name, tool_args)
                    steps.append(f"{tool_name}({args_display}) → success")
                except Exception as e:
                    result = f"Error: {e}"
                    steps.append(f"{tool_name}({args_display}) → failed")

                print(f"   → {result[:200]}...")

                # Offload tool output to TOOL_LOG table
                compact_result = memory_manager.write_tool_log(
                    thread_id, tc.id, tool_name, raw_args, str(result)
                )
                messages.append(
                    {"role": "tool", "tool_call_id": tc.id, "content": compact_result}
                )

                # If tools changed memory state, refresh context
                if tool_name in {
                    "search_tavily",
                    "summarize_conversation",
                    "summarize_and_store",
                }:
                    context = build_context()
                    if len(messages) >= 2 and messages[1].get("role") == "user":
                        messages[1]["content"] = context
                    usage = calculate_context_usage(context)
                    print(
                        f"   Refreshed context: {usage['percent']}% "
                        f"({usage['tokens']}/{usage['max']} tokens)"
                    )

            if timed_out:
                break
        else:
            final_answer = msg.content or ""
            print(f"\n✅ DONE ({len(steps)} tool calls)")
            break

    # ── Guarded stop ──
    if not final_answer:
        reason = "time limit" if timed_out else "iteration limit"
        print(f"\n⚠️ Stopped due to {reason}. Generating best-effort final answer...")
        try:
            final_messages = messages + [
                {
                    "role": "user",
                    "content": "Finalize your answer using the context and tool outputs so far. Do not call tools.",
                }
            ]
            final_resp = call_openai_chat(final_messages, tools=None)
            final_answer = final_resp.choices[0].message.content or ""
        except Exception as e:
            final_answer = f"Error: unable to finalize answer: {e}"

    # ── 6. Save results ──
    if steps:
        memory_manager.write_workflow(query, steps, final_answer)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=final_answer)
    except:
        pass
    memory_manager.write_conversational_memory(final_answer, "assistant", thread_id)

    print("\n" + "=" * 50 + f"\n💬 ANSWER:\n{final_answer}\n" + "=" * 50)
    return final_answer


--------

## Step 4: Run Proteus Through Research Scenarios

### Scenario 1: Simple Literature Query

In [ ]:
call_agent(
    "I'm looking for papers on transformer architectures applied to time-series "
    "forecasting. What's in our knowledge base?",
    thread_id="SESSION-2025-001",
)


### Scenario 2: Follow-up on the Same Session

Watch how Proteus uses conversational memory from the previous turn:

In [ ]:
call_agent(
    "Interesting — can you search the web for more recent work on that topic? "
    "I want to see what's been published in 2025 or 2026.",
    thread_id="SESSION-2025-001",
)


### Scenario 3: New Research Thread Requiring Web Search

In [ ]:
call_agent(
    "I've heard about a new technique called 'flow matching' for generative models. "
    "Can you find recent papers on it and summarize the key ideas?",
    thread_id="SESSION-2025-002",
)


### Scenario 4: Cross-Referencing Prior Research

Proteus should recall entities and papers from previous sessions:

In [ ]:
call_agent(
    "Remember the papers we found on transformer architectures? "
    "How do those relate to the flow matching approach?",
    thread_id="SESSION-2025-003",
)


### Visualize Context Window Growth

In [ ]:
import matplotlib.pyplot as plt

if context_size_history:
    tokens = [t for _, _, t in context_size_history]

    plt.figure(figsize=(8, 3))
    plt.plot(range(1, len(tokens) + 1), tokens, marker="o")
    plt.xlabel("Global Iteration (across all runs)")
    plt.ylabel("Estimated Tokens")
    plt.title("Context Window Size Over Agent Iterations")
    plt.tight_layout()
    plt.show()
else:
    print("No iterations recorded — run call_agent() first.")


--------

## Step 5: Baseline — The Naive Agent (No Context Engineering)

To appreciate the impact of memory and context engineering, let's see what happens **without them**.

`call_agent_naive` deliberately removes three key optimizations:

| Technique Removed | What Happens Instead | Effect on Context Window |
|---|---|---|
| **Tool output offloading** (`write_tool_log`) | Full raw tool outputs stay in messages | Each tool call adds thousands of tokens |
| **Summarization tools** | Excluded from tool list — agent can't compact context | Context only grows, never shrinks |
| **Context refresh after search** | No rebuild from memory after tool calls | Stale + bloated context persists |
| **Memory-backed context rebuild** | Messages persist as one flat list | Everything accumulates indefinitely |

### Why This Matters

In a real agent loop, the LLM is called **once per iteration** with the full `messages` list. Without offloading, every tool output ever produced sits in that list. After just 3 web searches, the context could grow by 10,000+ tokens — consuming budget that could be used for reasoning.

In [ ]:
naive_context_size_history = []
_naive_messages_by_thread = {}


def call_agent_naive(
    query: str,
    thread_id: str = "naive_1",
    dynamic_tools_override: list = None,
    max_iterations: int = 10,
    max_execution_time_s: float = 60.0,
) -> str:
    """Naive agent harness — NO context engineering.

    Differences from call_agent:
    1. Full raw tool outputs stay in messages (no write_tool_log offloading)
    2. No summarisation tools available (agent cannot compact context)
    3. No context refresh after memory-mutating tools
    4. Messages persist across calls — context only grows, never shrinks
    5. No memory reads — conversation history IS the raw messages list
    """
    thread_id = str(thread_id)
    steps = []
    start_time = time.time()
    timed_out = False

    # Get tools — but exclude summarisation tools
    if dynamic_tools_override is not None:
        dynamic_tools = dynamic_tools_override
    else:
        dynamic_tools = memory_manager.read_toolbox(query, k=5)
    dynamic_tools = [
        t
        for t in dynamic_tools
        if t.get("function", {}).get("name")
        not in {"summarize_conversation", "summarize_and_store", "expand_summary"}
    ]

    # No memory reads — raw messages list IS the only context
    if thread_id not in _naive_messages_by_thread:
        _naive_messages_by_thread[thread_id] = [
            {
                "role": "system",
                "content": "You are a Research Paper Assistant with access to tools.",
            }
        ]
    messages = _naive_messages_by_thread[thread_id]

    messages.append({"role": "user", "content": query})
    final_answer = ""

    tool_schema_chars = len(json_lib.dumps(dynamic_tools)) if dynamic_tools else 0
    tool_schema_tokens = tool_schema_chars // 4

    for iteration in range(max_iterations):
        msg_chars = sum(len(m.get("content", "") or "") for m in messages)
        naive_context_size_history.append((msg_chars // 4) + tool_schema_tokens)

        if max_execution_time_s and (time.time() - start_time) > max_execution_time_s:
            timed_out = True
            break

        response = call_openai_chat(messages, tools=dynamic_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": msg.content,
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": "function",
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments,
                            },
                        }
                        for tc in msg.tool_calls
                    ],
                }
            )
            for tc in msg.tool_calls:
                tool_args = json_lib.loads(tc.function.arguments or "{}")
                try:
                    result = execute_tool(tc.function.name, tool_args)
                    steps.append(f"{tc.function.name} → success")
                except Exception as e:
                    result = f"Error: {e}"
                    steps.append(f"{tc.function.name} → failed")

                # KEY DIFFERENCE: raw tool output goes straight into messages
                messages.append(
                    {"role": "tool", "tool_call_id": tc.id, "content": str(result)}
                )
        else:
            final_answer = msg.content or ""
            break

    if not final_answer:
        try:
            messages.append(
                {
                    "role": "user",
                    "content": "Finalize your answer. Do not call tools.",
                }
            )
            final_answer = (
                call_openai_chat(messages, tools=None).choices[0].message.content or ""
            )
        except Exception as e:
            final_answer = f"Error: {e}"

    messages.append({"role": "assistant", "content": final_answer})
    print(
        f"✅ Naive agent done ({len(steps)} tool calls, "
        f"{len(messages)} messages in context)"
    )
    return final_answer


--------

## Step 6: Head-to-Head Comparison

Run both agents through the same progressive queries — five escalating research questions that build on each other. This tests memory continuity, context management, and synthesis quality.

In [ ]:
import uuid

# Reset all trackers for a clean comparison
context_size_history.clear()
naive_context_size_history.clear()
_naive_messages_by_thread.clear()

eng_thread = str(uuid.uuid4())[:8]
naive_thread = str(uuid.uuid4())[:8]

# Progressive queries that build on each other
queries = [
    "Search for recent papers on AI agent memory published in 2025 or 2026.",
    "Pick the most interesting paper from those results and give me the key takeaways.",
    "What other approaches or viewpoints might that paper have missed?",
    "Summarize everything we've discussed so far in this session.",
    "What was the first question I asked in this conversation?",
]

for i, q in enumerate(queries, 1):
    print("=" * 60)
    print(f"QUERY {i}/5 — WITH CONTEXT ENGINEERING (thread: {eng_thread})")
    print(f"  >> {q}")
    print("=" * 60)
    call_agent(q, thread_id=eng_thread)
    print("\n")

for i, q in enumerate(queries, 1):
    print("=" * 60)
    print(f"QUERY {i}/5 — NAIVE / NO CONTEXT ENGINEERING (thread: {naive_thread})")
    print(f"  >> {q}")
    print("=" * 60)
    call_agent_naive(q, thread_id=naive_thread)
    print("\n")


### Visualize the Difference

In [ ]:
import matplotlib.pyplot as plt

eng_tokens = [t for _, _, t in context_size_history]
naive_tokens = naive_context_size_history

plt.figure(figsize=(9, 4))
if eng_tokens:
    plt.plot(
        range(1, len(eng_tokens) + 1),
        eng_tokens,
        marker="o",
        label="With Context/Memory Engineering",
    )
if naive_tokens:
    plt.plot(
        range(1, len(naive_tokens) + 1),
        naive_tokens,
        marker="s",
        label="Naive (no offloading/summarisation)",
    )
plt.xlabel("Iteration")
plt.ylabel("Estimated Tokens")
plt.title("Context Window Growth: Engineered vs Naive Agent")
plt.legend()
plt.tight_layout()
plt.show()


--------

## Key Takeaways

### Agent Architecture Concepts

In OpenAI-style framing:
- An **agent run** (one user turn handled) is what `call_agent(...)` executes
- Within a run, the **tool-call loop** repeats: model reasoning → optional tool calls → harness executes tools → model observes results → repeat until a final answer

An **agent harness** is the runtime scaffolding around that loop. In this workshop, it is a **memory-based agent harness** where:
- Context is assembled from multiple memory types each run
- Tools are discovered and executed within the run
- Outputs are written back into memory for future runs
- Summaries compact context while preserving continuity

### What Makes the Difference

| Aspect | Naive Agent | Proteus (Engineered) |
|--------|-------------|-------------------|
| **Context growth** | Unbounded — every tool output accumulates | Managed — tool outputs offloaded, conversations compacted |
| **Memory** | None — only raw message history | 6 specialized types with semantic search |
| **Tool discovery** | All tools all the time | Semantic retrieval of relevant tools per query |
| **Knowledge retention** | Lost between sessions | Persistent across research sessions |
| **Resolution patterns** | Rediscovered every time | Learned workflows reused automatically |

### The Practical Takeaway

Strong agents are not just model prompts. They are **run + harness systems**, and memory engineering is the control layer that makes them reliable, stateful, and scalable. The key discipline is deciding:
- What should be stored, retrieved, summarized, and refreshed
- How to keep context windows relevant, not just large
- How to treat memory as an evolving system that improves agent reliability over time

--------

## Workshop Complete 🎉

You've built a complete memory-powered AI agent from the ground up:

| Activity | What You Built |
|----------|---------------|
| **1** | Vector search foundations with OracleVS and arXiv research papers |
| **2** | Memory architecture — 6 types, SQL + vector stores, design principles |
| **3** | MemoryManager class and semantic Toolbox with LLM augmentation |
| **4** | Context engineering — usage tracking, summarization, JIT retrieval, web search |
| **5** | Agent harness, research scenarios, and engineered vs. naive comparison |

Proteus now demonstrates how modern AI agents maintain context, learn from interactions, and manage information across sessions — all backed by Oracle AI Database 26ai as the converged storage layer for relational, vector, and semantic data.